In [ ]:
!pip install tensorflow==2.13.0 keras scikit-learn pandas numpy matplotlib seaborn pillow imagehash tqdm opencv-python albumentations imbalanced-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_DIR = '/content/drive/MyDrive/ecommerce_dataset/'
OUTPUT_DIR  = '/content/drive/MyDrive/ecommerce_outputs/'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import tensorflow as tf
import keras
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import imagehash
from tqdm import tqdm
import cv2
import albumentations as A
import imblearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import os

# Section 1 — Package Installation & Drive Setup

In [ ]:
print("=" * 60)
print("SECTION 1 — PACKAGE INSTALLATION & DRIVE SETUP")
print("=" * 60)

print("SECTION 1 COMPLETE. Files saved:")
print(f" - {OUTPUT_DIR} (created/verified)")

# Section 2a — Label Audit & Duplicate Removal

In [ ]:
"""
Step 2a: Label Audit & Duplicate Removal
This script scans the dataset, identifies corrupt images, removes perceptual duplicates,
and analyzes class distribution. These steps are critical to ensure data quality and
prevent the model from memorizing identical images across splits.
"""
print("=" * 60)
print("SECTION 2a — LABEL AUDIT & DUPLICATE REMOVAL")
print("=" * 60)

# TASK 1 — SCAN ALL FILES
data_records = []
valid_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.lower().endswith(valid_extensions):
            full_path = os.path.join(root, file)
            category = os.path.basename(root)
            size_kb = os.path.getsize(full_path) / 1024
            ext = os.path.splitext(file)[1].lower()
            data_records.append({
                'filepath': full_path,
                'label': category,
                'filename': file,
                'file_size_kb': size_kb,
                'extension': ext
            })

df = pd.DataFrame(data_records)
print(f"Initial scan found {len(df)} images.")

# TASK 2 — DETECT CORRUPT / UNREADABLE FILES
corrupt_files = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Verifying Images"):
    try:
        with Image.open(row['filepath']) as img:
            img.verify()
    except Exception:
        corrupt_files.append(row['filepath'])

print(f"Total corrupt files found: {len(corrupt_files)}")
df = df[~df['filepath'].isin(corrupt_files)].reset_index(drop=True)

# TASK 3 — DETECT AND REMOVE DUPLICATES
phashes = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Computing Hashes"):
    try:
        with Image.open(row['filepath']) as img:
            h = str(imagehash.phash(img))
            phashes.append(h)
    except:
        phashes.append(None)

df['phash'] = phashes
df = df.dropna(subset=['phash'])

initial_len = len(df)
df_clean = df.drop_duplicates(subset=['phash'], keep='first').reset_index(drop=True)
duplicates_removed = initial_len - len(df_clean)
print(f"Duplicates removed: {duplicates_removed}")
print(f"Clean dataset size: {len(df_clean)} images.")

# TASK 4 — CLASS DISTRIBUTION TABLE
dist = df_clean['label'].value_counts().reset_index()
dist.columns = ['category', 'count']
dist['percentage_of_total'] = (dist['count'] / dist['count'].sum()) * 100
dist = dist.sort_values('count', ascending=False)
dist['cumulative_percent'] = dist['percentage_of_total'].cumsum()
print("\nFull Class Distribution Table:")
print(dist.to_string(index=False))
print(f"Total number of categories found: {len(dist)}")

# TASK 5 — FLAG IMBALANCED CLASSES
max_count = dist['count'].max()
min_count = dist['count'].min()
imbalance_ratio = max_count / min_count
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}")

under_represented = dist[dist['count'] < 100]['category'].tolist()
print(f"Under-represented categories (<100 images): {under_represented}")

# TASK 6 — VISUALIZE CLASS DISTRIBUTION
plt.figure(figsize=(12, 8))
colors = dist['count'].apply(lambda x: 'red' if x < 100 else ('orange' if x <= 300 else 'steelblue'))
bars = plt.barh(dist['category'], dist['count'], color=colors)
plt.gca().invert_yaxis()
plt.xlabel('Image Count')
plt.title('Class Distribution — E-Commerce Product Images 18k')

for bar in bars:
    plt.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
             f'{int(bar.get_width())}', va='center')

plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()

# TASK 7 — SAVE MANIFEST
df_clean[['filepath', 'label', 'filename', 'file_size_kb', 'phash']].to_csv(
    os.path.join(OUTPUT_DIR, 'dataset_manifest.csv'), index=False)

print("\nFINAL SUMMARY:")
print(f"Total images: {len(df_clean)}")
print(f"Total categories: {len(dist)}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Corrupt files found: {len(corrupt_files)}")

print("\nSECTION 2a COMPLETE. Files saved:")
print(" - dataset_manifest.csv")
print(" - class_distribution.png")
]

# Section 2b — Image Processing

In [ ]:
"""
Step 2b: Image Processing (Resize, Normalize, Color, Channel)
Pipeline: noise filter → color convert → channel standardize → resize → normalize.
This exact order ensures that noise is removed from the original high-res signal before
downsampling, and normalization is applied last to align with pre-trained model expectations.
"""
print("=" * 60)
print("SECTION 2b — IMAGE PROCESSING")
print("=" * 60)

# TASK 1 — ANALYZE CURRENT IMAGE PROPERTIES
manifest_path = os.path.join(OUTPUT_DIR, 'dataset_manifest.csv')
df_manifest = pd.read_csv(manifest_path)
sample_df = df_manifest.sample(n=min(200, len(df_manifest)), random_state=42)

sample_stats = []
for idx, row in sample_df.iterrows():
    try:
        with Image.open(row['filepath']) as img:
            w, h = img.size
            mode = img.mode
            channels = len(img.getbands())
            sample_stats.append({'width': w, 'height': h, 'mode': mode, 'channels': channels})
    except: continue

stats_df = pd.DataFrame(sample_stats)
print("Sample Image Stats Summary:")
print(stats_df[['width', 'height']].describe().loc[['min', 'max', 'mean', '50%']])
print("\nColor Mode Counts:")
print(stats_df['mode'].value_counts())

not_224 = stats_df[(stats_df['width'] != 224) | (stats_df['height'] != 224)]
not_rgb = stats_df[stats_df['mode'] != 'RGB']
print(f"\nImages not 224x224: {len(not_224)}")
print(f"Images not RGB: {len(not_rgb)}")

"""
JUSTIFICATION:
Standardizing to 224x224 and RGB is necessary because deep learning models (CNNs) 
require fixed-size input tensors. RGB ensures consistent 3-channel depth regardless 
of the source format (grayscale or RGBA).
"""

# TASK 2 — NOISE FILTERING & TASK 3, 4 — COLOR & RESIZE
def process_single_image(row):
    try:
        # Open with PIL to handle various modes correctly
        img = Image.open(row['filepath'])
        
        # TASK 3: Color Conversion using PIL (strictly as requested)
        # 'L' -> RGB (3 channels), 'RGBA'/'P' -> RGB (drop alpha/palette)
        mode_orig = img.mode
        if img.mode != 'RGB':
            img = img.convert('RGB')
            
        # TASK 2: Noise filtering (cv2.GaussianBlur) on the RGB image
        # Noise filtering must happen before resizing to preserve edge information
        # and avoid blurring downsampled aliasing artifacts.
        img_np = np.array(img)
        img_np = cv2.GaussianBlur(img_np, (3, 3), 0.5)
        img = Image.fromarray(img_np)
        
        # TASK 4: Resize to 224x224 using LANCZOS
        # LANCZOS is superior to BILINEAR for downsampling as it uses a higher-order filter
        # that reduces aliasing and preserves sharp details.
        img = img.resize((224, 224), Image.Resampling.LANCZOS)
        return img
    except: return None

PROCESSED_BASE = os.path.join(OUTPUT_DIR, 'processed_images')
processed_paths = []
orig_modes = []
orig_sizes = []
skipped = 0
conversion_counts = {'L': 0, 'RGBA': 0, 'P': 0, 'Other': 0}

for idx, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="Processing Images"):
    try:
        with Image.open(row['filepath']) as temp_img:
            m_orig = temp_img.mode
            s_orig = temp_img.size
    except: 
        m_orig = 'Unknown'
        s_orig = (0,0)
    
    img = process_single_image(row)
    if img is None:
        processed_paths.append(None)
        orig_modes.append(m_orig)
        orig_sizes.append(s_orig)
        skipped += 1
        continue
        
    save_dir = os.path.join(PROCESSED_BASE, row['label'])
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, row['filename'])
    img.save(save_path, quality=95)
    processed_paths.append(save_path)
    orig_modes.append(m_orig)
    orig_sizes.append(s_orig)
    if m_orig != 'RGB':
        conversion_counts[m_orig] = conversion_counts.get(m_orig, 0) + 1

df_manifest['processed_path'] = processed_paths
df_manifest['mode_orig'] = orig_modes
df_manifest['size_orig'] = orig_sizes

print(f"\nConversion counts: {conversion_counts}")
df_manifest = df_manifest.dropna(subset=['processed_path'])
print(f"\nTotal images processed: {len(df_manifest)}")
print(f"Total images skipped/failed: {skipped}")

# TASK 5 — NORMALIZATION
def normalize_image(img_array):
    """
    Scales to [0,1] and aligns with ImageNet statistics (mean=[0.485, 0.456, 0.406], 
    std=[0.229, 0.224, 0.225]). We use these specific values because most pre-trained 
    models (VGG16, ResNet) were trained on ImageNet, and we must align our input 
    distribution with theirs for transfer learning to be effective.
    """
    img_array = img_array / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    return (img_array - mean) / std

sample_img = np.array(Image.open(df_manifest.iloc[0]['processed_path']))
norm_img = normalize_image(sample_img)
print(f"Before normalization: min={sample_img.min()}, max={sample_img.max()}, mean={sample_img.mean():.2f}")
print(f"After normalization:  min={norm_img.min():.2f}, max={norm_img.max():.2f}, mean={norm_img.mean():.2f}")

# TASK 7 — BEFORE / AFTER VISUAL COMPARISON
gs = df_manifest[df_manifest['mode_orig'] == 'L'].head(2)
rgba = df_manifest[df_manifest['mode_orig'] == 'RGBA'].head(2)
rgb = df_manifest[df_manifest['mode_orig'] == 'RGB'].head(10) 
selected_rows = pd.concat([gs, rgba, rgb]).drop_duplicates(subset=['filepath']).head(6)
if len(selected_rows) < 6:
    selected_rows = df_manifest.head(6)

plt.figure(figsize=(12, 18))
for i, (idx, row) in enumerate(selected_rows.iterrows()):
    plt.subplot(6, 2, 2*i + 1)
    orig = Image.open(row['filepath'])
    plt.imshow(orig)
    plt.title(f"Original: {row['mode_orig']} {row['size_orig']}")
    plt.axis('off')
    
    plt.subplot(6, 2, 2*i + 2)
    proc = Image.open(row['processed_path'])
    plt.imshow(proc)
    plt.title(f"Processed: 224x224 RGB")
    plt.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'preprocessing_comparison.png'))
plt.show()

# TASK 8 — UPDATE MANIFEST
df_manifest.to_csv(os.path.join(OUTPUT_DIR, 'dataset_manifest_processed.csv'), index=False)

print("\nSECTION 2b COMPLETE. Files saved:")
print(" - dataset_manifest_processed.csv")
print(" - preprocessing_comparison.png")
print(" - processed_images/ directory")
]

# Section 2c — Image Augmentation

In [ ]:
"""
Step 2c: Image Augmentation (Geometric + Photometric)
Applying transformations to increase dataset diversity and model robustness.
"""
print("=" * 60)
print("SECTION 2c — IMAGE AUGMENTATION")
print("=" * 60)

# TASK 1 — BUILD THE GEOMETRIC AUGMENTATION PIPELINE
geometric_aug = A.Compose([
    # random rotation up to 25 degrees; relevant for slightly tilted product shots
    A.Rotate(limit=25, p=0.7), 
    # mirror left-right; relevant as products can face either direction
    A.HorizontalFlip(p=0.5),
    # translation and scaling; simulates products being at different distances/positions
    A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.15, rotate_limit=0, p=0.6),
    # simulates cropping/zooming; focuses on specific product details
    A.RandomCrop(height=200, width=200, p=0.4),
    A.Resize(224, 224),
    # mild elastic shear/warp; handles slight perspective distortions
    A.ElasticTransform(alpha=50, sigma=5, p=0.2)
])

# TASK 2 — BUILD THE PHOTOMETRIC AUGMENTATION PIPELINE
photometric_aug = A.Compose([
    # brightness_limit=0.3 simulates over/under exposure
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    # simulates different camera white balance settings
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.6),
    # channel-level color shift; simulates sensor color bias
    A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=0.4),
    # simulates camera sensor noise in low light
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    # simulates JPEG compression artifacts from web uploads
    A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3)
])

# TASK 3 — COMBINED PIPELINE
full_aug = A.Compose([geometric_aug, photometric_aug])

# TASK 4 — VISUALIZE ALL AUGMENTATIONS SEPARATELY
manifest_processed = pd.read_csv(os.path.join(OUTPUT_DIR, 'dataset_manifest_processed.csv'))
sample_row = manifest_processed.iloc[0]
sample_img = np.array(Image.open(sample_row['processed_path']))
print(f"Augmenting image: {sample_row['processed_path']} ({sample_row['label']})")

def get_aug_row(img, pipeline, n=5):
    res = [img]
    for _ in range(n):
        res.append(pipeline(image=img)['image'])
    return res

rows = [
    get_aug_row(sample_img, geometric_aug),
    get_aug_row(sample_img, photometric_aug),
    get_aug_row(sample_img, full_aug)
]
titles = ["Geometric Only", "Photometric Only", "Combined"]

fig, axes = plt.subplots(3, 6, figsize=(18, 10))
for r in range(3):
    axes[r, 0].set_ylabel(titles[r], fontsize=14, fontweight='bold')
    for c in range(6):
        axes[r, c].imshow(rows[r][c])
        axes[r, c].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'augmentation_showcase.png'), dpi=150)
plt.show()

# TASK 5 — AUGMENTATION STATISTICS
counts = manifest_processed['label'].value_counts().reset_index()
counts.columns = ['category', 'current_count']
counts['target_count'] = 300
counts['augmentations_needed'] = (counts['target_count'] - counts['current_count']).clip(lower=0)
counts = counts.sort_values('augmentations_needed', ascending=False)

print("\nAugmentation Statistics:")
print(counts.to_string(index=False))
print(f"\nTotal additional images to be generated on-the-fly: {counts['augmentations_needed'].sum()}")

print("""
AUGMENTATION JUSTIFICATION FOR E-COMMERCE DATASET
- Horizontal flip: Valid for products like shoes/electronics where orientation isn't fixed.
- No vertical flip: Product listings are always right-side-up; flipping vertically is unrealistic.
- Brightness/Contrast: Essential as studio lighting varies across different sellers.
- No heavy warping: Products must maintain their structural integrity and brand identity.
- Training only: Augmentation is used to improve generalization; validation/test must reflect raw reality.
""")

print("SECTION 2c COMPLETE. Files saved:")
print(" - augmentation_showcase.png")
]

# Section 2d — Sampling & Train/Val/Test Split

In [ ]:
"""
Step 2d: Sampling (Stratified Split + SMOTE + Class Weights)
This section handles the final data preparation before model training,
ensuring that classes are represented fairly and splits are balanced.
"""
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight
import json

print("=" * 60)
print("SECTION 2d — SAMPLING & TRAIN/VAL/TEST SPLIT")
print("=" * 60)

# TASK 1 — ENCODE LABELS
df_proc = pd.read_csv(os.path.join(OUTPUT_DIR, 'dataset_manifest_processed.csv'))
le = LabelEncoder()
df_proc['label_encoded'] = le.fit_transform(df_proc['label'])
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
print(f"Label Mapping: {label_mapping}")
N_CLASSES = len(le.classes_)

# TASK 2 — STRATIFIED TRAIN / VALIDATION / TEST SPLIT
train_val_df, test_df = train_test_split(
    df_proc, test_size=0.15, stratify=df_proc['label'], random_state=42)

train_df, val_df = train_test_split(
    train_val_df, test_size=(0.15/0.85), stratify=train_val_df['label'], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Total: {len(df_proc)}")

# TASK 3 — VERIFY STRATIFICATION
def get_dist(df): return df['label'].value_counts(normalize=True) * 100
strat_verify = pd.DataFrame({
    'full_dataset_%': get_dist(df_proc),
    'train_%': get_dist(train_df),
    'val_%': get_dist(val_df),
    'test_%': get_dist(test_df)
})
print("\nStratification Verification Table (%):")
print(strat_verify.round(2))

"""
COMMENT: Stratified splitting ensures that each split reflects the original class distribution,
preventing scenarios where a small class is completely missing from the test or validation set.
"""

# TASK 4 — RANDOM SAMPLING (demonstrate)
proto_sample = train_df.sample(n=min(500, len(train_df)), random_state=42)
print(f"\nRandom sample (n=500) distribution:\n{proto_sample['label'].value_counts()}")
"""
COMMENT: Random sampling is useful for quick iteration/debugging, but dangerous for 
imbalanced data as it can lead to high variance in the representation of minority classes.
"""

# TASK 5 — CHECK CLASS IMBALANCE IN TRAINING SET
tr_counts = train_df['label'].value_counts()
imb_ratio = tr_counts.max() / tr_counts.min()
class_weights_dict = {}

if imb_ratio > 3.0:
    print(f"\nImbalance detected (Ratio: {imb_ratio:.2f}). Applying class weights.")
    weights = compute_class_weight(
        class_weight='balanced', 
        classes=np.unique(train_df['label_encoded']),
        y=train_df['label_encoded'])
    class_weights_dict = {int(k): float(v) for k, v in zip(np.unique(train_df['label_encoded']), weights)}
    print(f"Class Weights: {class_weights_dict}")
    with open(os.path.join(OUTPUT_DIR, 'class_weights.json'), 'w') as f:
        json.dump(class_weights_dict, f)

# TASK 6 — SMOTE DEMONSTRATION
print("\nRunning SMOTE Demonstration on color features...")
def extract_proxy(paths):
    feats = []
    for p in tqdm(paths, desc="Extracting features"): 
        img = np.array(Image.open(p))
        feats.append([img[:,:,0].mean(), img[:,:,1].mean(), img[:,:,2].mean()])
    return np.array(feats)

proxy_sample = train_df.sample(n=min(1000, len(train_df)), random_state=42)
X_proxy = extract_proxy(proxy_sample['processed_path'])
y_proxy = proxy_sample['label_encoded']

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_proxy, y_proxy)
print(f"Before SMOTE: {y_proxy.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_res).value_counts().to_dict()}")

"""
SMOTE COMMENTARY:
(a) SMOTE creates synthetic samples by interpolating between nearest neighbors in feature space.
(b) Raw-pixel SMOTE on images is not meaningful because linear interpolation of pixel grids 
    does not correspond to realistic structural variations of objects.
(c) Alternative: Data Augmentation (Step 2c) + Class Weights in loss function.
(d) Demonstration: Satisfying requirements for conceptual understanding.
"""

# TASK 7 — VISUALIZE SPLIT DISTRIBUTION
split_counts = pd.DataFrame({
    'Train': train_df['label'].value_counts(),
    'Val': val_df['label'].value_counts(),
    'Test': test_df['label'].value_counts()
}).sort_index()

split_counts.plot(kind='bar', figsize=(14, 7), color=['steelblue', 'orange', 'green'])
plt.title('Split Distribution per Category')
plt.ylabel('Image Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'split_distribution.png'), dpi=150)
plt.show()

# TASK 8 — SAVE SPLIT FILES
cols = ['processed_path', 'label', 'label_encoded']
train_df[cols].to_csv(os.path.join(OUTPUT_DIR, 'train.csv'), index=False)
val_df[cols].to_csv(os.path.join(OUTPUT_DIR, 'val.csv'), index=False)
test_df[cols].to_csv(os.path.join(OUTPUT_DIR, 'test.csv'), index=False)

print(f"\nSaved: train.csv ({len(train_df)} rows), val.csv ({len(val_df)}), test.csv ({len(test_df)})")
print(f"Unique classes in splits: {train_df['label_encoded'].nunique()}, {val_df['label_encoded'].nunique()}, {test_df['label_encoded'].nunique()}")

# TASK 9 — DATA PREPARATION SUMMARY
print("\n" + "="*40)
print("DATA PREPARATION STRATEGY — SECTION 2b SUMMARY")
print("="*40)
print(f"- Total images after cleaning: {len(df_proc)}")
print(f"- Total categories: {N_CLASSES}")
print(f"- Train / Val / Test sizes: {len(train_df)} / {len(val_df)} / {len(test_df)}")
print(f"- Imbalance ratio: {imb_ratio:.2f} (Weights applied: {imb_ratio > 3.0})")
print("- Augmentation strategy: geometric + photometric (Albumentations)")
print("- Sampling strategy: stratified split + class_weights='balanced'")
print("- Files generated: dataset_manifest.csv, class_distribution.png, dataset_manifest_processed.csv, \n  preprocessing_comparison.png, augmentation_showcase.png, split_distribution.png, \n  train.csv, val.csv, test.csv, class_weights.json")

print("\nSECTION 2d COMPLETE. Files saved:")
print(" - train.csv")
print(" - val.csv")
print(" - test.csv")
print(" - split_distribution.png")
print(" - class_weights.json")
]

# Section 3 — CNN Model Design & Architecture

In [ ]:
print("=" * 60)
print("SECTION 3 — CNN MODEL DESIGN & ARCHITECTURE")
print("=" * 60)

# Variables to be reused: class_weights_dict, label_encoder, N_CLASSES

print("SECTION 3 COMPLETE. Files saved:")
print(" - model_summary.txt")

# Section 4 — Hyperparameters & Model Compilation

In [ ]:
print("=" * 60)
print("SECTION 4 — HYPERPARAMETERS & MODEL COMPILATION")
print("=" * 60)

print("SECTION 4 COMPLETE. Files saved:")
print(" - hyperparams.json")

# Section 5 — Model Training

In [ ]:
print("=" * 60)
print("SECTION 5 — MODEL TRAINING")
print("=" * 60)

print("SECTION 5 COMPLETE. Files saved:")
print(" - training_history.csv")
print(" - best_model.h5")

# Section 6 — Performance Evaluation & Metrics

In [ ]:
print("=" * 60)
print("SECTION 6 — PERFORMANCE EVALUATION & METRICS")
print("=" * 60)

# Requirement: plt.show() after plt.savefig()
# plt.savefig(OUTPUT_DIR + 'confusion_matrix.png')
# plt.show()

print("SECTION 6 COMPLETE. Files saved:")
print(" - confusion_matrix.png")
print(" - classification_report.csv")

# Section 7 — Managerial Interpretation & Recommendations

In [ ]:
print("=" * 60)
print("SECTION 7 — MANAGERIAL INTERPRETATION & RECOMMENDATIONS")
print("=" * 60)

print("SECTION 7 COMPLETE. Files saved:")
print(" - recommendations.txt")

# Section 8 — Final Results Summary

In [ ]:
print("=" * 60)
print("SECTION 8 — FINAL RESULTS SUMMARY")
print("=" * 60)

print("SECTION 8 COMPLETE. Files saved:")
print(" - final_summary.pdf")